# Incremental Data Loading

In [0]:
%sql
create database sales;

In [0]:
%sql
-- DROP DATABASE sales
DROP DATABASE salesDWH

In [0]:
%sql
create table sales.orders
(
  OrderID INT, 
  OrderDate date, 
  CustomerID INT, 
  CustomerName varchar(50), 
  CustomerEmail varchar(50), 
  ProductID INT, 
  ProductName varchar(50), 
  ProductCategory varchar(50), 
  RegionID INT, 
  RegionName varchar(50), 
  Country varchar(50), 
  Quantity INT, 
  UnitPrice decimal(10,2), 
  TotalAmount decimal(10,2)
)

In [0]:
dbutils.fs.ls('dbfs:/user/hive/warehouse')

Out[3]: [FileInfo(path='dbfs:/user/hive/warehouse/sales.db/', name='sales.db/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/user/hive/warehouse/salesdwh.db/', name='salesdwh.db/', size=0, modificationTime=0)]

In [0]:
%sql
INSERT INTO sales.orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount)
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00), 
(2, '2024-02-02', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00), 
(3, '2024-02-03', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00), 
(4, '2024-02-04', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
(5, '2024-02-05', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00), 
(6, '2024-02-06', 102, 'Bob Smith', 'bob@example.com', 206, 'Smartwatch', 'Electronics', 303, 'Asia', 'China', 2, 200.00, 400.00),
(7, '2024-02-07', 105, 'Eve Adams', 'eve@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'Canada', 1, 800.00, 800.00), 
(8, '2024-02-08', 106, 'Frank Miller', 'frank@example.com', 207, 'Monitor', 'Accessories', 302, 'Europe', 'Italy', 2, 250.00, 500.00), 
(9, '2024-02-09', 107, 'Grace White', 'grace@example.com', 208, 'Keyboard', 'Accessories', 303, 'Asia', 'Japan', 3, 100.00, 300.00), 
(10, '2024-02-10', 104, 'David Lee', 'idavid@example.com', 209, 'Mouse', 'Accessories', 301, 'North America', 'USA', 1, 50.00, 50.00);

num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
INSERT INTO sales.orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount) 
VALUES 
(11, '2024-02-11', 108, 'Hannah Green', 'hannah@example.com', 210, 'Wireless Earbuds', 'Accessories', 302, 'Europe', 'Spain', 2, 120.00, 240.00), 
(12, '2024-02-12', 109, 'Ian Black', 'ian@example.com', 201, 'Laptop', 'Electronics', 303, 'Asia', 'India', 1, 800.00, 800.00), 
(13, '2024-02-13', 105, 'Eve Adams', 'eve@example.com', 202, 'Smartphone', 'Electronics', 301, 'North America', 'Canada', 1, 500.00, 500.00), 
(14, '2024-02-14', 110, 'Jack Wilson', 'jack@example.com', 211, 'External Hard Drive', 'Accessories', 302, 'Europe', 'UK', 2, 150.00, 300.00), 
(15, '2024-02-15', 101, 'Alice Johnson', 'alice@example.com', 203, 'Tablet', 'Electronics', 301, 'North America', 'USA', 1, 300.00, 300.00);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
select * from sales.orders

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,ProductID,ProductName,ProductCategory,RegionID,RegionName,Country,Quantity,UnitPrice,TotalAmount
1,2024-02-01,101,Alice Johnson,alice@example.com,201,Laptop,Electronics,301,North America,USA,2,800.00,1600.00
2,2024-02-02,102,Bob Smith,bob@example.com,202,Smartphone,Electronics,302,Europe,Germany,1,500.00,500.00
3,2024-02-03,103,Charlie Brown,charlie@example.com,203,Tablet,Electronics,303,Asia,India,3,300.00,900.00
4,2024-02-04,101,Alice Johnson,alice@example.com,204,Headphones,Accessories,301,North America,USA,1,150.00,150.00
5,2024-02-05,104,David Lee,david@example.com,205,Gaming Console,Electronics,302,Europe,France,1,400.00,400.00
6,2024-02-06,102,Bob Smith,bob@example.com,206,Smartwatch,Electronics,303,Asia,China,2,200.00,400.00
7,2024-02-07,105,Eve Adams,eve@example.com,201,Laptop,Electronics,301,North America,Canada,1,800.00,800.00
8,2024-02-08,106,Frank Miller,frank@example.com,207,Monitor,Accessories,302,Europe,Italy,2,250.00,500.00
9,2024-02-09,107,Grace White,grace@example.com,208,Keyboard,Accessories,303,Asia,Japan,3,100.00,300.00
10,2024-02-10,104,David Lee,idavid@example.com,209,Mouse,Accessories,301,North America,USA,1,50.00,50.00


# Data Warehousing

In [0]:
%sql
create database salesDWH

### Staging Layer

In [0]:
%sql
--Initial Load
create table salesDWH.stg_sales
as
select * from sales.orders



num_affected_rows,num_inserted_rows


### once you get new record we have to truncate the table everytime

In [0]:
%sql

create or replace table salesDWH.stg_sales
as
select * from sales.orders
where OrderDate > '2024-02-10'

num_affected_rows,num_inserted_rows


### Transformation

In [0]:
%sql
create or replace view salesDWH.trans_sales
as
select * from salesDWH.stg_sales where Quantity is not null

In [0]:
%sql
select * from salesDWH.trans_sales

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,ProductID,ProductName,ProductCategory,RegionID,RegionName,Country,Quantity,UnitPrice,TotalAmount
1,2024-02-01,101,Alice Johnson,alice@example.com,201,Laptop,Electronics,301,North America,USA,2,800.00,1600.00
2,2024-02-02,102,Bob Smith,bob@example.com,202,Smartphone,Electronics,302,Europe,Germany,1,500.00,500.00
3,2024-02-03,103,Charlie Brown,charlie@example.com,203,Tablet,Electronics,303,Asia,India,3,300.00,900.00
4,2024-02-04,101,Alice Johnson,alice@example.com,204,Headphones,Accessories,301,North America,USA,1,150.00,150.00
5,2024-02-05,104,David Lee,david@example.com,205,Gaming Console,Electronics,302,Europe,France,1,400.00,400.00
6,2024-02-06,102,Bob Smith,bob@example.com,206,Smartwatch,Electronics,303,Asia,China,2,200.00,400.00
7,2024-02-07,105,Eve Adams,eve@example.com,201,Laptop,Electronics,301,North America,Canada,1,800.00,800.00
8,2024-02-08,106,Frank Miller,frank@example.com,207,Monitor,Accessories,302,Europe,Italy,2,250.00,500.00
9,2024-02-09,107,Grace White,grace@example.com,208,Keyboard,Accessories,303,Asia,Japan,3,100.00,300.00
10,2024-02-10,104,David Lee,idavid@example.com,209,Mouse,Accessories,301,North America,USA,1,50.00,50.00


## Core Layer

In [0]:
%sql
create or replace table salesDWH.core_sales
(
  OrderID INT, 
  OrderDate date, 
  CustomerID INT, 
  CustomerName varchar(50), 
  CustomerEmail varchar(50), 
  ProductID INT, 
  ProductName varchar(50), 
  ProductCategory varchar(50), 
  RegionID INT, 
  RegionName varchar(50), 
  Country varchar(50), 
  Quantity INT, 
  UnitPrice decimal(10,2), 
  TotalAmount decimal(10,2)
)

In [0]:
%sql
insert into salesDWH.core_sales
select * from salesDWH.trans_sales

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
select * from salesDWH.core_sales

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,ProductID,ProductName,ProductCategory,RegionID,RegionName,Country,Quantity,UnitPrice,TotalAmount
11,2024-02-11,108,Hannah Green,hannah@example.com,210,Wireless Earbuds,Accessories,302,Europe,Spain,2,120.00,240.00
12,2024-02-12,109,Ian Black,ian@example.com,201,Laptop,Electronics,303,Asia,India,1,800.00,800.00
13,2024-02-13,105,Eve Adams,eve@example.com,202,Smartphone,Electronics,301,North America,Canada,1,500.00,500.00
14,2024-02-14,110,Jack Wilson,jack@example.com,211,External Hard Drive,Accessories,302,Europe,UK,2,150.00,300.00
15,2024-02-15,101,Alice Johnson,alice@example.com,203,Tablet,Electronics,301,North America,USA,1,300.00,300.00


#Dimensional Modeling

In [0]:
%sql
create database sales_new;

In [0]:
%sql
create table sales_new.orders
(
  OrderID INT, 
  OrderDate date, 
  CustomerID INT, 
  CustomerName varchar(50), 
  CustomerEmail varchar(50), 
  ProductID INT, 
  ProductName varchar(50), 
  ProductCategory varchar(50), 
  RegionID INT, 
  RegionName varchar(50), 
  Country varchar(50), 
  Quantity INT, 
  UnitPrice decimal(10,2), 
  TotalAmount decimal(10,2)
)

In [0]:
%sql
INSERT INTO sales_new.orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount)
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00), 
(2, '2024-02-02', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00), 
(3, '2024-02-03', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00), 
(4, '2024-02-04', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
(5, '2024-02-05', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00), 
(6, '2024-02-06', 102, 'Bob Smith', 'bob@example.com', 206, 'Smartwatch', 'Electronics', 303, 'Asia', 'China', 2, 200.00, 400.00),
(7, '2024-02-07', 105, 'Eve Adams', 'eve@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'Canada', 1, 800.00, 800.00), 
(8, '2024-02-08', 106, 'Frank Miller', 'frank@example.com', 207, 'Monitor', 'Accessories', 302, 'Europe', 'Italy', 2, 250.00, 500.00), 
(9, '2024-02-09', 107, 'Grace White', 'grace@example.com', 208, 'Keyboard', 'Accessories', 303, 'Asia', 'Japan', 3, 100.00, 300.00), 
(10, '2024-02-10', 104, 'David Lee', 'david@example.com', 209, 'Mouse', 'Accessories', 301, 'North America', 'USA', 1, 50.00, 50.00);

num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
select * from  sales_new.orders

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,ProductID,ProductName,ProductCategory,RegionID,RegionName,Country,Quantity,UnitPrice,TotalAmount
1,2024-02-01,101,Alice Johnson,alice@example.com,201,Laptop,Electronics,301,North America,USA,2,800.00,1600.00
2,2024-02-02,102,Bob Smith,bob@example.com,202,Smartphone,Electronics,302,Europe,Germany,1,500.00,500.00
3,2024-02-03,103,Charlie Brown,charlie@example.com,203,Tablet,Electronics,303,Asia,India,3,300.00,900.00
4,2024-02-04,101,Alice Johnson,alice@example.com,204,Headphones,Accessories,301,North America,USA,1,150.00,150.00
5,2024-02-05,104,David Lee,david@example.com,205,Gaming Console,Electronics,302,Europe,France,1,400.00,400.00
6,2024-02-06,102,Bob Smith,bob@example.com,206,Smartwatch,Electronics,303,Asia,China,2,200.00,400.00
7,2024-02-07,105,Eve Adams,eve@example.com,201,Laptop,Electronics,301,North America,Canada,1,800.00,800.00
8,2024-02-08,106,Frank Miller,frank@example.com,207,Monitor,Accessories,302,Europe,Italy,2,250.00,500.00
9,2024-02-09,107,Grace White,grace@example.com,208,Keyboard,Accessories,303,Asia,Japan,3,100.00,300.00
10,2024-02-10,104,David Lee,david@example.com,209,Mouse,Accessories,301,North America,USA,1,50.00,50.00


#DWH

In [0]:
%sql
create database orderDWH

### Staging layer

In [0]:
%sql
create or replace table orderDWH.stg_sales
as
select * from sales_new.orders

num_affected_rows,num_inserted_rows


### Transformation

In [0]:
%sql
create or replace view orderDWH.trans_sales
as
select * from orderDWH.stg_sales where Quantity is not null

In [0]:
%sql
select * from orderDWH.trans_sales

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,ProductID,ProductName,ProductCategory,RegionID,RegionName,Country,Quantity,UnitPrice,TotalAmount
1,2024-02-01,101,Alice Johnson,alice@example.com,201,Laptop,Electronics,301,North America,USA,2,800.00,1600.00
2,2024-02-02,102,Bob Smith,bob@example.com,202,Smartphone,Electronics,302,Europe,Germany,1,500.00,500.00
3,2024-02-03,103,Charlie Brown,charlie@example.com,203,Tablet,Electronics,303,Asia,India,3,300.00,900.00
4,2024-02-04,101,Alice Johnson,alice@example.com,204,Headphones,Accessories,301,North America,USA,1,150.00,150.00
5,2024-02-05,104,David Lee,david@example.com,205,Gaming Console,Electronics,302,Europe,France,1,400.00,400.00
6,2024-02-06,102,Bob Smith,bob@example.com,206,Smartwatch,Electronics,303,Asia,China,2,200.00,400.00
7,2024-02-07,105,Eve Adams,eve@example.com,201,Laptop,Electronics,301,North America,Canada,1,800.00,800.00
8,2024-02-08,106,Frank Miller,frank@example.com,207,Monitor,Accessories,302,Europe,Italy,2,250.00,500.00
9,2024-02-09,107,Grace White,grace@example.com,208,Keyboard,Accessories,303,Asia,Japan,3,100.00,300.00
10,2024-02-10,104,David Lee,david@example.com,209,Mouse,Accessories,301,North America,USA,1,50.00,50.00


In [0]:
%sql
select CustomerID, count(CustomerID) as count from orderDWH.trans_sales group by CustomerID having count>1

CustomerID,count
101,2
102,2
104,2


#### DimCustomer

In [0]:
%sql
create or replace table orderDWH.DimCustomer
(
  CustomerID INT,
  CustomerName STRING,
  CustomerEmail STRING,
  DimCustomerKey INT
)

In [0]:
%sql
create or replace view orderDWH.view_DimCustomer
as
select T.*, row_number() over(order by T.CustomerID) as DimCustomerKey from (
select
  distinct (CustomerID) as CustomerID,
  CustomerName,
  CustomerEmail
from orderDWH.trans_sales
) as T

In [0]:
%sql
select * from orderDWH.view_DimCustomer

CustomerID,CustomerName,CustomerEmail,DimCustomerKey
101,Alice Johnson,alice@example.com,1
102,Bob Smith,bob@example.com,2
103,Charlie Brown,charlie@example.com,3
104,David Lee,david@example.com,4
105,Eve Adams,eve@example.com,5
106,Frank Miller,frank@example.com,6
107,Grace White,grace@example.com,7


In [0]:
%sql
insert into orderDWH.DimCustomer
select * from orderDWH.view_DimCustomer

num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
select * from orderDWH.DimCustomer

CustomerID,CustomerName,CustomerEmail,DimCustomerKey
101,Alice Johnson,alice@example.com,1
102,Bob Smith,bob@example.com,2
103,Charlie Brown,charlie@example.com,3
104,David Lee,david@example.com,4
105,Eve Adams,eve@example.com,5
106,Frank Miller,frank@example.com,6
107,Grace White,grace@example.com,7


### DimProduct

In [0]:
%sql
create or replace table orderDWH.DimProduct
(
  ProductID INT,
  ProductName STRING,
  ProductCategory STRING,
  DimProductKey INT
)

In [0]:
%sql
create or replace view orderDWH.view_DimProduct
as
select T.*, row_number() over(order by T.ProductID) as DimProductKey from (
select
  distinct (ProductID) as ProductID,
  ProductName,
  ProductCategory
from orderDWH.trans_sales
) as T

In [0]:
%sql
select * from orderDWH.view_DimProduct

ProductID,ProductName,ProductCategory,DimProductKey
201,Laptop,Electronics,1
202,Smartphone,Electronics,2
203,Tablet,Electronics,3
204,Headphones,Accessories,4
205,Gaming Console,Electronics,5
206,Smartwatch,Electronics,6
207,Monitor,Accessories,7
208,Keyboard,Accessories,8
209,Mouse,Accessories,9


In [0]:
%sql
insert into orderDWH.DimProduct
select * from orderDWH.view_DimProduct

num_affected_rows,num_inserted_rows
9,9


In [0]:
%sql
select * from orderDWH.DimProduct

ProductID,ProductName,ProductCategory,DimProductKey
201,Laptop,Electronics,1
202,Smartphone,Electronics,2
203,Tablet,Electronics,3
204,Headphones,Accessories,4
205,Gaming Console,Electronics,5
206,Smartwatch,Electronics,6
207,Monitor,Accessories,7
208,Keyboard,Accessories,8
209,Mouse,Accessories,9


### DimRegion

In [0]:
%sql
create or replace table orderDWH.DimRegion
(
  RegionID INT,
  RegionName STRING,
  Country STRING,
  DimRegionKey INT
)

In [0]:
%sql
create or replace view orderDWH.view_DimRegion
as
select T.*, row_number() over(order by T.RegionID) as DimRegionKey from (
select
  distinct (RegionID) as RegionID,
  RegionName,
  Country
from orderDWH.trans_sales
) as T

In [0]:
%sql
insert into orderDWH.DimRegion
select * from orderDWH.view_DimRegion

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
select * from orderDWH.DimRegion

RegionID,RegionName,Country,DimRegionKey
301,North America,Canada,1
301,North America,USA,2
302,Europe,Germany,3
302,Europe,France,4
302,Europe,Italy,5
303,Asia,India,6
303,Asia,China,7
303,Asia,Japan,8


### DimDate

In [0]:
%sql
create or replace table orderDWH.DimDate
(
  OrderDate Date,
  DimDateKey INT
)

In [0]:
%sql
create or replace view orderDWH.view_DimDate
as
select T.*, row_number() over(order by T.OrderDate) as DimDateKey from (
select
  distinct (OrderDate) as OrderDate
from orderDWH.trans_sales
) as T

In [0]:
%sql
insert into orderDWH.DimDate
select * from orderDWH.view_DimDate

num_affected_rows,num_inserted_rows
10,10


### Fact Table

In [0]:
%sql
CREATE OR REPLACE TABLE orderDWH.FactSales
(
  OrderID INT, 
  Quantity DECIMAL, 
  UnitPrice DECIMAL, 
  TotalAmount DECIMAL,
  DimProductKey INT,
  DimCustomerKey INT,
  DimRegionKey INT,
  DimDateKey INT
)

In [0]:
%sql
select
  F.OrderID,
  F.Quantity,
  F.UnitPrice,
  F.TotalAmount,
  DC.DimCustomerKey,
  DP.DimProductKey,
  DR.DimRegionKey,
  DD.DimDateKey
from
  orderDWH.trans_sales F
left join
  orderDWH.DimCustomer DC
  on F.CustomerID = DC.CustomerID
left join
  orderDWH.DimProduct DP
  on F.ProductID = DP.ProductID
left join
  orderDWH.DimRegion DR
  on F.Country = DR.Country
left join
  orderDWH.DimDate DD
  on F.OrderDate = DD.OrderDate



OrderID,Quantity,UnitPrice,TotalAmount,DimCustomerKey,DimProductKey,DimRegionKey,DimDateKey
1,2,800.00,1600.00,1,1,2,1
2,1,500.00,500.00,2,2,3,2
3,3,300.00,900.00,3,3,6,3
4,1,150.00,150.00,1,4,2,4
5,1,400.00,400.00,4,5,4,5
6,2,200.00,400.00,2,6,7,6
7,1,800.00,800.00,5,1,1,7
8,2,250.00,500.00,6,7,5,8
9,3,100.00,300.00,7,8,8,9
10,1,50.00,50.00,4,9,2,10


# Slowly Changing Dimension(SCD)

In [0]:
%sql
create database sales_scd

In [0]:
%sql
create table sales_scd.orders
(
  OrderID INT, 
  OrderDate date, 
  CustomerID INT, 
  CustomerName varchar(50), 
  CustomerEmail varchar(50), 
  ProductID INT, 
  ProductName varchar(50), 
  ProductCategory varchar(50), 
  RegionID INT, 
  RegionName varchar(50), 
  Country varchar(50), 
  Quantity INT, 
  UnitPrice decimal(10,2), 
  TotalAmount decimal(10,2)
)

In [0]:
%sql
INSERT INTO sales_scd.orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount)
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00), 
(2, '2024-02-02', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00), 
(3, '2024-02-03', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00), 
(4, '2024-02-04', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
(5, '2024-02-05', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00), 
(6, '2024-02-06', 102, 'Bob Smith', 'bob@example.com', 206, 'Smartwatch', 'Electronics', 303, 'Asia', 'China', 2, 200.00, 400.00),
(7, '2024-02-07', 105, 'Eve Adams', 'eve@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'Canada', 1, 800.00, 800.00), 
(8, '2024-02-08', 106, 'Frank Miller', 'frank@example.com', 207, 'Monitor', 'Accessories', 302, 'Europe', 'Italy', 2, 250.00, 500.00), 
(9, '2024-02-09', 107, 'Grace White', 'grace@example.com', 208, 'Keyboard', 'Accessories', 303, 'Asia', 'Japan', 3, 100.00, 300.00), 
(10, '2024-02-10', 104, 'David Lee', 'david@example.com', 209, 'Mouse', 'Accessories', 301, 'North America', 'USA', 1, 50.00, 50.00);

num_affected_rows,num_inserted_rows
10,10


## SCD TYPE-1

In [0]:
%sql
select * from sales_scd.orders

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,ProductID,ProductName,ProductCategory,RegionID,RegionName,Country,Quantity,UnitPrice,TotalAmount
1,2024-02-01,101,Alice Johnson,alice@example.com,201,Laptop,Electronics,301,North America,USA,2,800.00,1600.00
2,2024-02-02,102,Bob Smith,bob@example.com,202,Smartphone,Electronics,302,Europe,Germany,1,500.00,500.00
3,2024-02-03,103,Charlie Brown,charlie@example.com,203,Tablet,Electronics,303,Asia,India,3,300.00,900.00
4,2024-02-04,101,Alice Johnson,alice@example.com,204,Headphones,Accessories,301,North America,USA,1,150.00,150.00
5,2024-02-05,104,David Lee,david@example.com,205,Gaming Console,Electronics,302,Europe,France,1,400.00,400.00
6,2024-02-06,102,Bob Smith,bob@example.com,206,Smartwatch,Electronics,303,Asia,China,2,200.00,400.00
7,2024-02-07,105,Eve Adams,eve@example.com,201,Laptop,Electronics,301,North America,Canada,1,800.00,800.00
8,2024-02-08,106,Frank Miller,frank@example.com,207,Monitor,Accessories,302,Europe,Italy,2,250.00,500.00
9,2024-02-09,107,Grace White,grace@example.com,208,Keyboard,Accessories,303,Asia,Japan,3,100.00,300.00
10,2024-02-10,104,David Lee,david@example.com,209,Mouse,Accessories,301,North America,USA,1,50.00,50.00


In [0]:
%sql
create or replace view sales_scd.view_DimProducts
AS
select distinct(ProductID) as ProductID, ProductName, ProductCategory
from sales_scd.orders
where OrderDate > '2024-02-10'

In [0]:
%sql
create or replace table sales_scd.DimProducts
(
  ProductID INT,
  ProductName string,
  ProductCategory string
)

In [0]:
%sql
insert into sales_scd.DimProducts
select ProductID,ProductName,ProductCategory from sales_scd.view_DimProducts

num_affected_rows,num_inserted_rows
9,9


In [0]:
%sql
select * from sales_scd.DimProducts


ProductID,ProductName,ProductCategory
203,Tablet,Electronics
208,Keyboard,Accessories
205,Gaming Console,Electronics
209,Mouse,Accessories
204,Headphones,Accessories
206,Smartwatch,Electronics
207,Monitor,Accessories
202,Smartphone,Electronics
201,Laptop,Electronics


In [0]:
%sql
INSERT INTO sales_scd.orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount)
VALUES 
(1, '2024-02-11', 101, 'Alice Johnson', 'alice@example.com', 201, 'Gaming Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00), 
(2, '2024-02-12', 102, 'Bob Smith', 'bob@example.com', 230, 'Airpods', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00)

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
select * from  sales_scd.view_DimProducts

ProductID,ProductName,ProductCategory
201,Gaming Laptop,Electronics
230,Airpods,Electronics


## MERGE - SCD TYPE -1

In [0]:
%sql
merge into sales_scd.DimProducts as trg
using sales_scd.view_DimProducts as src
on trg.ProductID = src.ProductID
when matched then update set *
when not matched then insert *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
select * from sales_scd.DimProducts

ProductID,ProductName,ProductCategory
203,Tablet,Electronics
208,Keyboard,Accessories
205,Gaming Console,Electronics
209,Mouse,Accessories
204,Headphones,Accessories
206,Smartwatch,Electronics
207,Monitor,Accessories
202,Smartphone,Electronics
201,Gaming Laptop,Electronics
230,Airpods,Electronics
